In [16]:
import os
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from concurrent.futures import ThreadPoolExecutor

from sklearn.metrics import classification_report, accuracy_score
from sklearn.linear_model import SGDClassifier
from sklearn.model_selection import ParameterGrid
from sklearn.metrics import log_loss

In [2]:
def load_or_cache(split, folder, csv_path, cache_dir="cache", size=(224, 224), batch=1000, workers=8):
    os.makedirs(cache_dir, exist_ok=True)
    xp = os.path.join(cache_dir, f"X_{split}_{size[0]}.npy")
    yp = os.path.join(cache_dir, f"y_{split}_{size[0]}.npy")

    if os.path.exists(xp) and os.path.exists(yp):
        return np.load(xp), np.load(yp)

    df = pd.read_csv(csv_path, usecols=["filename", "category_id"])
    files = df["filename"].astype(str).values
    y = df["category_id"].values.astype(int)
    
    def read(f):
        path = os.path.join(folder, f)
        img = cv2.imread(path)

        if img is None:
            raise ValueError(f"Could not read image: {path}")

        return cv2.resize(img, size, interpolation=cv2.INTER_AREA).ravel()

    X = np.empty((len(files), size[0] * size[1] * 3), np.uint8)
    with ThreadPoolExecutor(workers) as ex:
        for i in range(0, len(files), batch):
            X[i:i + batch] = list(ex.map(read, files[i:i + batch]))

    np.save(xp, X)
    np.save(yp, y)
    return X, y

In [3]:
folder = "dataset_updated/224x224"

In [4]:
X_train, y_train = load_or_cache("train", folder, "train/listings_train.csv")    

In [5]:
X_val, y_val = load_or_cache("val", folder, "validation/listings_val.csv")  

In [6]:
X_test, y_test = load_or_cache("test", folder, "test/listings_test.csv")

In [7]:
def extract_features(flat, size=224, bins=8):
    img = flat.reshape(size, size, 3)

    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)
    c = cv2.calcHist([hsv], [0, 1, 2], None, [bins]*3,
                     [0, 180, 0, 256, 0, 256]).ravel()
    c /= max(c.sum(), 1)

    return np.sqrt(c).astype(np.float32)

In [8]:
def load_or_cache_features(split, X, extract_features, workers=8, cache_dir="cache", feature_name="color_histogram"):
    os.makedirs(cache_dir, exist_ok=True)
    xp = os.path.join(cache_dir, f"X_{split}_{feature_name}.npy")

    if os.path.exists(xp):
        print(f"Loading cached {split} features...")
        return np.load(xp)

    print(f"Extracting {split} features...")
    with ThreadPoolExecutor(max_workers=workers) as ex:
        X_features = np.array(list(ex.map(extract_features, X)), dtype=np.float32)

    np.save(xp, X_features)
    print(f"Saved {split} features to {xp}")
    return X_features

In [9]:
X_train_features = load_or_cache_features("train", X_train, extract_features)
X_val_features = load_or_cache_features("val", X_val, extract_features)
X_test_features = load_or_cache_features("test", X_test, extract_features)

Loading cached train features...
Loading cached val features...
Loading cached test features...


In [12]:
model = SGDClassifier(loss = 'log_loss', eta0 = 0.01, learning_rate = 'constant', random_state = 1, verbose = 0)

In [13]:
model.fit(X_train_features, y_train)

,"<a class=""param-doc-link"" style=""anchor-name: --doc-link-loss;"" rel=""noreferrer"" target=""_blank"" href=""https://scikit-learn.org/1.9/modules/generated/sklearn.linear_model.SGDClassifier.html#:~:text=loss,-%7B%27hinge%27%2C%20%27log_loss%27%2C%20%27modified_huber%27%2C%20%27squared_hinge%27%2C%20%20%20%20%20%20%20%20%27perceptron%27%2C%20%27squared_error%27%2C%20%27huber%27%2C%20%27epsilon_insensitive%27%2C%20%20%20%20%20%20%20%20%27squared_epsilon_insensitive%27%7D%2C%20default%3D%27hinge%27""> loss loss: {'hinge', 'log_loss', 'modified_huber', 'squared_hinge', 'perceptron', 'squared_error', 'huber', 'epsilon_insensitive', 'squared_epsilon_insensitive'}, default='hinge'The loss function to be used.- 'hinge' gives a linear SVM.- 'log_loss' gives logistic regression, a probabilistic classifier.- 'modified_huber' is another smooth loss that brings tolerance to outliers as well as probability estimates.- 'squared_hinge' is like hinge but is quadratically penalized.- 'perceptron' is the linear loss used by the perceptron algorithm.- The other losses, 'squared_error', 'huber', 'epsilon_insensitive' and 'squared_epsilon_insensitive' are designed for regression but can be useful in classification as well; see :class:`~sklearn.linear_model.SGDRegressor` for a description.More details about the losses formulas can be found in the :ref:`User Guide<sgd_mathematical_formulation>` and you can find a visualisation of the lossfunctions in:ref:`sphx_glr_auto_examples_linear_model_plot_sgd_loss_functions.py`.",'log_loss'
,"random_state random_state: int, RandomState instance, default=NoneUsed for shuffling the data, when ``shuffle`` is set to ``True``.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.Integer values must be in the range `[0, 2**32 - 1]`.",1
,"learning_rate learning_rate: str, default='optimal'The learning rate schedule:- 'constant': `eta = eta0`- 'optimal': `eta = 1.0 / (alpha * (t + t0))` where `t0` is chosen by a heuristic proposed by Leon Bottou.- 'invscaling': `eta = eta0 / pow(t, power_t)`- 'adaptive': `eta = eta0`, as long as the training keeps decreasing. Each time n_iter_no_change consecutive epochs fail to decrease the training loss by tol or fail to increase validation score by tol if `early_stopping` is `True`, the current learning rate is divided by 5.- 'pa1': passive-aggressive algorithm 1, see [1]_. Only with `loss='hinge'`. Update is `w += eta y x` with `eta = min(eta0, loss/||x||**2)`.- 'pa2': passive-aggressive algorithm 2, see [1]_. Only with `loss='hinge'`. Update is `w += eta y x` with `eta = hinge_loss / (||x||**2 + 1/(2 eta0))`... versionadded:: 0.20 Added 'adaptive' option... versionadded:: 1.8 Added options 'pa1' and 'pa2'",'constant'
,"penalty penalty: {'l2', 'l1', 'elasticnet', None}, default='l2'The penalty (aka regularization term) to be used. Defaults to 'l2'which is the standard regularizer for linear SVM models. 'l1' and'elasticnet' might bring sparsity to the model (feature selection)not achievable with 'l2'. No penalty is added when set to `None`.You can see a visualisation of the penalties in:ref:`sphx_glr_auto_examples_linear_model_plot_sgd_penalties.py`.",'l2'
,"alpha alpha: float, default=0.0001Constant that multiplies the regularization term. The higher thevalue, the stronger the regularization. Also used to compute thelearning rate when `learning_rate` is set to 'optimal'.Values must be in the range `[0.0, inf)`.",0.0001
,"l1_ratio l1_ratio: float, default=0.15The Elastic Net mixing parameter, with 0 <= l1_ratio <= 1.l1_ratio=0 corresponds to L2 penalty, l1_ratio=1 to L1.Only used if `penalty` is 'elasticnet'.Values must be in the range `[0.0, 1.0]` or can be `None` if`penalty` is not `elasticnet`... versionchanged:: 1.7 `l1_ratio` can be `None` when `penalty` is not ""elasticnet"".",0.15
,"fit_intercept fit_intercept: bool, default=TrueWhether the intercept should be estimated or not. If False, thedata is assumed to be already 

In [14]:
y_train_pred = model.predict(X_train_features)
train_accuracy = accuracy_score(y_train, y_train_pred)
print(train_accuracy)

In [17]:
y_train_proba = model.predict_proba(X_train_features)
print("Train loss:", log_loss(y_train, y_train_proba))

Train loss: 3.149148464202881


In [18]:
model = SGDClassifier()

In [20]:
model.get_params()

{'alpha': 0.0001,
 'average': False,
 'class_weight': None,
 'early_stopping': False,
 'epsilon': 0.1,
 'eta0': 0.01,
 'fit_intercept': True,
 'l1_ratio': 0.15,
 'learning_rate': 'optimal',
 'loss': 'hinge',
 'max_iter': 1000,
 'n_iter_no_change': 5,
 'n_jobs': None,
 'penalty': 'l2',
 'power_t': 0.5,
 'random_state': None,
 'shuffle': True,
 'tol': 0.001,
 'validation_fraction': 0.1,
 'verbose': 0,
 'warm_start': False}

In [23]:
hyperparameters = {
    "loss": ["log_loss"],
    "learning_rate": ["constant"],
    "eta0": [0.0001, 0.001, 0.01, 0.1],
    "alpha": [1e-6, 1e-5, 1e-4, 1e-3, 1e-2],
    "random_state": [1]
}

In [24]:
results = []
best_accuracy = 0
best_loss = np.inf
best_params = None

In [25]:
for param in ParameterGrid(hyperparameters):
    print(param)
    model.set_params(**param)
    model.fit(X_train_features, y_train)

    val_accuracy = accuracy_score(y_val, model.predict(X_val_features))
    val_loss = log_loss(y_val, model.predict_proba(X_val_features),
                        labels=model.classes_)
    print(f"  val_accuracy={val_accuracy:.4f}  val_loss={val_loss:.4f}")

    results.append({**param, "val_accuracy": val_accuracy, "val_loss": val_loss})

    if val_accuracy > best_accuracy:
        best_accuracy = val_accuracy
        best_loss = val_loss
        best_params = param.copy()

{'alpha': 1e-06, 'eta0': 0.0001, 'learning_rate': 'constant', 'loss': 'log_loss', 'random_state': 1}
  val_accuracy=0.0972  val_loss=3.3750
{'alpha': 1e-06, 'eta0': 0.001, 'learning_rate': 'constant', 'loss': 'log_loss', 'random_state': 1}
  val_accuracy=0.0990  val_loss=3.2825
{'alpha': 1e-06, 'eta0': 0.01, 'learning_rate': 'constant', 'loss': 'log_loss', 'random_state': 1}
  val_accuracy=0.1183  val_loss=3.1561
{'alpha': 1e-06, 'eta0': 0.1, 'learning_rate': 'constant', 'loss': 'log_loss', 'random_state': 1}
  val_accuracy=0.1154  val_loss=3.1768
{'alpha': 1e-05, 'eta0': 0.0001, 'learning_rate': 'constant', 'loss': 'log_loss', 'random_state': 1}
  val_accuracy=0.0972  val_loss=3.3750
{'alpha': 1e-05, 'eta0': 0.001, 'learning_rate': 'constant', 'loss': 'log_loss', 'random_state': 1}
  val_accuracy=0.0990  val_loss=3.2825
{'alpha': 1e-05, 'eta0': 0.01, 'learning_rate': 'constant', 'loss': 'log_loss', 'random_state': 1}
  val_accuracy=0.1183  val_loss=3.1573
{'alpha': 1e-05, 'eta0': 0.1,

In [27]:
results_df = pd.DataFrame(results)

top_5 = results_df.sort_values(by="val_accuracy",ascending=False).head(5).reset_index(drop=True)
print(top_5)

      alpha  eta0 learning_rate      loss  random_state  val_accuracy  \
0  0.000100  0.01      constant  log_loss             1      0.118908   
1  0.001000  0.01      constant  log_loss             1      0.118692   
2  0.000001  0.01      constant  log_loss             1      0.118259   
3  0.000010  0.01      constant  log_loss             1      0.118259   
4  0.000010  0.10      constant  log_loss             1      0.118042   

   val_loss  
0  3.165784  
1  3.236745  
2  3.156096  
3  3.157305  
4  3.171552  


In [30]:
model = SGDClassifier(**best_params)

In [31]:
model.fit(X_test_features, y_test)

,"<a class=""param-doc-link"" style=""anchor-name: --doc-link-loss;"" rel=""noreferrer"" target=""_blank"" href=""https://scikit-learn.org/1.9/modules/generated/sklearn.linear_model.SGDClassifier.html#:~:text=loss,-%7B%27hinge%27%2C%20%27log_loss%27%2C%20%27modified_huber%27%2C%20%27squared_hinge%27%2C%20%20%20%20%20%20%20%20%27perceptron%27%2C%20%27squared_error%27%2C%20%27huber%27%2C%20%27epsilon_insensitive%27%2C%20%20%20%20%20%20%20%20%27squared_epsilon_insensitive%27%7D%2C%20default%3D%27hinge%27""> loss loss: {'hinge', 'log_loss', 'modified_huber', 'squared_hinge', 'perceptron', 'squared_error', 'huber', 'epsilon_insensitive', 'squared_epsilon_insensitive'}, default='hinge'The loss function to be used.- 'hinge' gives a linear SVM.- 'log_loss' gives logistic regression, a probabilistic classifier.- 'modified_huber' is another smooth loss that brings tolerance to outliers as well as probability estimates.- 'squared_hinge' is like hinge but is quadratically penalized.- 'perceptron' is the linear loss used by the perceptron algorithm.- The other losses, 'squared_error', 'huber', 'epsilon_insensitive' and 'squared_epsilon_insensitive' are designed for regression but can be useful in classification as well; see :class:`~sklearn.linear_model.SGDRegressor` for a description.More details about the losses formulas can be found in the :ref:`User Guide<sgd_mathematical_formulation>` and you can find a visualisation of the lossfunctions in:ref:`sphx_glr_auto_examples_linear_model_plot_sgd_loss_functions.py`.",'log_loss'
,"random_state random_state: int, RandomState instance, default=NoneUsed for shuffling the data, when ``shuffle`` is set to ``True``.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.Integer values must be in the range `[0, 2**32 - 1]`.",1
,"learning_rate learning_rate: str, default='optimal'The learning rate schedule:- 'constant': `eta = eta0`- 'optimal': `eta = 1.0 / (alpha * (t + t0))` where `t0` is chosen by a heuristic proposed by Leon Bottou.- 'invscaling': `eta = eta0 / pow(t, power_t)`- 'adaptive': `eta = eta0`, as long as the training keeps decreasing. Each time n_iter_no_change consecutive epochs fail to decrease the training loss by tol or fail to increase validation score by tol if `early_stopping` is `True`, the current learning rate is divided by 5.- 'pa1': passive-aggressive algorithm 1, see [1]_. Only with `loss='hinge'`. Update is `w += eta y x` with `eta = min(eta0, loss/||x||**2)`.- 'pa2': passive-aggressive algorithm 2, see [1]_. Only with `loss='hinge'`. Update is `w += eta y x` with `eta = hinge_loss / (||x||**2 + 1/(2 eta0))`... versionadded:: 0.20 Added 'adaptive' option... versionadded:: 1.8 Added options 'pa1' and 'pa2'",'constant'
,"penalty penalty: {'l2', 'l1', 'elasticnet', None}, default='l2'The penalty (aka regularization term) to be used. Defaults to 'l2'which is the standard regularizer for linear SVM models. 'l1' and'elasticnet' might bring sparsity to the model (feature selection)not achievable with 'l2'. No penalty is added when set to `None`.You can see a visualisation of the penalties in:ref:`sphx_glr_auto_examples_linear_model_plot_sgd_penalties.py`.",'l2'
,"alpha alpha: float, default=0.0001Constant that multiplies the regularization term. The higher thevalue, the stronger the regularization. Also used to compute thelearning rate when `learning_rate` is set to 'optimal'.Values must be in the range `[0.0, inf)`.",0.0001
,"l1_ratio l1_ratio: float, default=0.15The Elastic Net mixing parameter, with 0 <= l1_ratio <= 1.l1_ratio=0 corresponds to L2 penalty, l1_ratio=1 to L1.Only used if `penalty` is 'elasticnet'.Values must be in the range `[0.0, 1.0]` or can be `None` if`penalty` is not `elasticnet`... versionchanged:: 1.7 `l1_ratio` can be `None` when `penalty` is not ""elasticnet"".",0.15
,"fit_intercept fit_intercept: bool, default=TrueWhether the intercept should be estimated or not. If False, thedata is assumed to be already 

In [32]:
y_test_pred = model.predict(X_test_features)
test_accuracy = accuracy_score(y_test, y_test_pred)
print(test_accuracy)

0.12859649122807018


In [33]:
y_test_proba = model.predict_proba(X_test_features)
print("Test loss:", log_loss(y_test, y_test_proba))

Test loss: 3.2299129962921143


In [35]:
print("Train Accuracy", train_accuracy)
print("Val Accuracy", best_accuracy)
print("Test Accuracy", test_accuracy)

Train Accuracy 0.1292312965978233
Val Accuracy 0.1189083820662768
Test Accuracy 0.12859649122807018


### AI Usage

* Used Claude to find a way to use GPU in training the model
* Used Claude to suggest on handling overfitting